# Lesson 14 · Tokens & embeddings: turning text into a sequence of vectors

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/14-token-embedding-en.ipynb)

This notebook accompanies [Lesson 14 of Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/token-embedding/), a deep-learning course originally in
Vietnamese. It is the English edition of the course's notebook
([Vietnamese original](https://github.com/trungnguyenvn/hoc-sau-notebooks/blob/main/14-token-embedding.ipynb)):
the code, the data and every check are the same. The reel and the interactive demo are on the lesson page.
A neural network takes only numbers. In pure Python and numpy, with no tokenizer library, we write both of GPT-2's
steps that turn text into numbers ourselves: the **byte-level BPE tokenizer** (text → IDs) and the **embedding
table** (ID → vector), on real parallel translated sentences (FLORES-200) and on actual rows of GPT-2's trained
embedding table. The lesson compares Vietnamese, the course's original language, with English: the Vietnamese text is
what the tokenizer reads and what the numbers measure, so it stays as it is, with an English translation alongside.

**What you'll do:**
1. Turn text into UTF-8 bytes and cut sentences into “chunks” with GPT-2's pre-tokenizer.
2. Learn a BPE tokenizer from scratch: count adjacent pairs, merge the most frequent pair, repeat.
3. Rewrite GPT-2's encoder with its exact 50,000 merges, and measure why Vietnamese costs 4.52 times as many tokens.
4. Look up rows of the embedding table, measure cosine similarity and find nearest neighbors.
5. Try the analogy $b - a + c$ (king − man + woman), with and without excluding the input words.

**Background:** [Lesson 03 · Multilayer networks & the XOR problem](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/mlp-va-xor/)
(the layer $z = xW + b$) and [Lesson 06 · Backpropagation](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/lan-truyen-nguoc/).
**Run time:** under a minute on a CPU, including the first FLORES-200 download (~26 MB).

Every number in the notebook is computed by the cells below (or read from the data files the demo uses), and
matches the numbers in the demo.

> The **🤔 Predict first** cells ask a question before you run something. Answer it yourself, then run the next cell.

## 1 · Setup

Only numpy and matplotlib are needed (Colab has them preinstalled); the tokenization is pure Python. The function
`num()` prints numbers in the English format: a decimal point, and commas between thousands. That is the same point
Python uses when you *type* a decimal (`0.1`); printing only adds the thousands separators and a real minus sign.

In [ ]:
import base64
import hashlib
import itertools
import json
import math
import re
import sys
import tarfile
import unicodedata
import urllib.request
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter

# The course's palette (colorblind-safe), always paired with a second encoding (marker shape, hatching, label)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
DIVERGING = LinearSegmentedColormap.from_list("orange_gray_blue", [ORANGE, "#f2f2f2", BLUE])  # negative → 0 → positive
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def num(x, digits=2):
    """A number in the English format: 13,007 · 0.1 · −4.38 (never “−0”)."""
    s = f"{round(float(x), digits) + 0.0:,.{digits}f}"
    return s.replace("-", "−")


def pct(p, digits=2):
    return num(100 * p, digits) + "%"


def tick(v, _pos=None):
    """An axis tick label: 0.5 · −2 · 60,000 (no trailing zeros)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return num(v, len(s.split(".")[1]) if "." in s else 0)


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def log_tick(v, _pos=None):
    """A log-axis tick label: 0.5 · 20 · 10⁻¹²."""
    if 0.01 <= v <= 1000:
        return tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def fmt_axes(fig):
    """Put the tick labels of every axis of the figure (colorbars included) in the English format; call right before
    plt.show(). A log axis gets 1–2–5 ticks if it is short, or powers of 10 (evenly spaced, at most ~9 ticks) if it is
    long."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                # only axes still on matplotlib's default number formatter: text labels and custom formatters stay
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
                if step > 1:
                    axis.set_minor_locator(NullLocator())
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))


print(num(50257, 0), num(4.5213, 2), pct(0.480412))

## 2 · The data

Three sources, each checked by its MD5 hash, so everyone who runs it gets exactly the same data:
- **FLORES-200** (NLLB Team, 2022; license CC BY-SA 4.0): the same set of English sentences, professionally
  translated into 200 languages. We use the English and Vietnamese versions: the *dev* split (997 sentences) to learn
  BPE ourselves, the *devtest* split (1,012 sentences) to measure. The cell below downloads the original (~26 MB)
  from Meta's server.
- **`gpt2-bpe.json`**: GPT-2's table of 50,000 merges, as pairs of IDs, together with the demo's sample sentences and
  the numbers of tokens that three tokenizers count on FLORES-200.
- **`gpt2-embed.json`**: 137 real rows of the small GPT-2's embedding table (English words that have a token of their
  own), compressed to int8, together with the neighbors and analogies computed on all 50,257 tokens. The whole table
  (154 MB in float32) is too big to download.

The two JSON files are exactly the files the demo on the lesson page uses.

In [ ]:
FLORES = ("https://dl.fbaipublicfiles.com/nllb/flores200_dataset.tar.gz", "ceca3bc3a69207c4ae8427266a0b6514")
DATA = Path("data")


def fetch_flores():
    DATA.mkdir(exist_ok=True)
    p = DATA / "flores200_dataset.tar.gz"
    if not p.exists():
        urllib.request.urlretrieve(FLORES[0], p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == FLORES[1], (
        f"{p.name} is corrupted; delete it and download it again"
    )
    return p


# The web demo's data files. When publishing, build.py replaces the string "COURSE_DATA_URL" with the data/ folder of
# the public notebook repo.
def fetch_json(name, md5):
    p = DATA / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"
    return json.loads(p.read_text())


flores_path = fetch_flores()
bpe_json = fetch_json("gpt2-bpe.json", "2cd95a0c22ae42cdeda5d6541f4f6f04")  # merge table + counts on FLORES
emb = fetch_json("gpt2-embed.json", "a55436ca8087ea20ef8fb8bdcaa99b15")  # 137 rows of the embedding table E
print("loaded:", flores_path.name, "· gpt2-bpe.json · gpt2-embed.json")

In [ ]:
with tarfile.open(flores_path) as tar:

    def read(split, lang):
        f = tar.extractfile(f"./flores200_dataset/{split}/{lang}.{split}")
        return f.read().decode("utf-8").rstrip("\n").split("\n")

    dev = {"en": read("dev", "eng_Latn"), "vi": read("dev", "vie_Latn")}
    test = {"en": read("devtest", "eng_Latn"), "vi": read("devtest", "vie_Latn")}

print(
    f"dev: {num(len(dev['vi']), 0)} sentences · devtest: {num(len(test['vi']), 0)} sentences "
    "(each split has both English and Vietnamese)"
)
for lang in ("en", "vi"):
    print(f"  devtest[0], {lang}: {test[lang][0]}")
assert len(dev["en"]) == len(dev["vi"]) == 997 and len(test["en"]) == len(test["vi"]) == bpe_json["flores"]["n"] == 1012

## 3 · Text → bytes: UTF-8

A computer stores text as Unicode **code points** (ô is U+00F4), and stores code points as **bytes** using UTF-8: an
English letter takes 1 byte, most Vietnamese letters with diacritics take 2 or 3. A byte has 256 values, so the
smallest “alphabet” that can read *any* text has just 256 symbols. GPT-2 starts from there: the 256 bytes are its
first 256 tokens.

The cell below prints the number of bytes of a few characters, of a sentence, and of the whole devtest split in both
languages.

In [ ]:
for ch in ["a", "ô", "đ", "ệ", "😀"]:
    b = ch.encode("utf-8")
    print(f"{ch!r}: code point U+{ord(ch):04X} → {len(b)} byte{'s' if len(b) > 1 else ''}: {b.hex(' ').upper()}")

s = "Tôi yêu Việt Nam."  # Vietnamese for “I love Vietnam.”
print(f"\n{s!r}: {len(s)} characters → {len(s.encode('utf-8'))} bytes")
n_char = {lang: sum(len(x) for x in test[lang]) for lang in test}
n_byte = {lang: sum(len(x.encode("utf-8")) for x in test[lang]) for lang in test}
for lang in ("en", "vi"):
    print(f"devtest {lang}: {num(n_char[lang], 0)} characters → {num(n_byte[lang], 0)} bytes")
print(
    f"the Vietnamese version has {pct(n_char['vi'] / n_char['en'] - 1)} more characters, "
    f"but {pct(n_byte['vi'] / n_byte['en'] - 1)} more bytes"
)

With the same content, the Vietnamese version has only 5.02% more characters than the English one but 38.99% more
bytes, because each letter with diacritics takes 2–3 bytes. The 3-byte letters (ạ, ả, ấ… ỹ) all begin with `E1 BA` or
`E1 BB`; remember these two bytes, we will meet them again in section 7.

A letter on screen can also have **two byte sequences**. Unicode lets you write “ệ” *precomposed* (NFC: one code
point) or *decomposed* (NFD: the letter e, then the dot below, then the circumflex, each mark a code point of its
own). Vietnamese text on the web has both.

In [ ]:
for form in ("NFC", "NFD"):
    t = unicodedata.normalize(form, "ệ")
    print(
        f"{form}: {len(t)} code point{'s' if len(t) > 1 else ''} {[f'U+{ord(c):04X}' for c in t]} "
        f"→ {len(t.encode('utf-8'))} bytes"
    )

## 4 · Cutting a sentence into chunks: GPT-2's pre-tokenizer

Before merging bytes, GPT-2 cuts a sentence into **chunks** with a regular expression, and no merge crosses a chunk
boundary. The GPT-2 paper explains: without this step, BPE learns many versions of the same word (`dog.`, `dog!`,
`dog?`), wasting room in the vocabulary. GPT-2's original expression is

```
's|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+
```

which means: the English contraction suffixes; a **run of letters** (with at most one space in front); a run of
digits; a run of other symbols; and whitespace. `\p{L}` is “any Unicode letter” and `\p{N}` is “any numeric
character”.

Python's `re` module doesn't understand `\p{L}`. Rather than approximate it, we build exactly those two character
classes from Python's Unicode table (`unicodedata.category`: categories starting with L are letters, those starting
with N are numbers), and write `\s` as exactly Unicode's whitespace characters. (Python's `\s` also matches the 4
control characters `\x1c`–`\x1f`, which Unicode doesn't consider whitespace.)

In [ ]:
def char_class(pred):
    """A character class for re: every code point that satisfies pred, grouped into ranges a-b."""
    parts, start = [], None
    for cp in range(sys.maxunicode + 2):
        ok = cp <= sys.maxunicode and pred(chr(cp))
        if ok and start is None:
            start = cp
        elif not ok and start is not None:
            a, b = re.escape(chr(start)), re.escape(chr(cp - 1))
            parts.append(a if cp - 1 == start else f"{a}-{b}")
            start = None
    return "".join(parts), len(parts)


L, n_L = char_class(lambda c: unicodedata.category(c).startswith("L"))  # \p{L}: letters
N, n_N = char_class(lambda c: unicodedata.category(c).startswith("N"))  # \p{N}: numbers
WS_CHARS = [*range(9, 14), 32, 0x85, 0xA0, 0x1680, *range(0x2000, 0x200B), 0x2028, 0x2029, 0x202F, 0x205F, 0x3000]
WS = re.escape("".join(map(chr, WS_CHARS)))  # Unicode's 25 whitespace characters (the White_Space property)
PAT = re.compile(rf"'s|'t|'re|'ve|'m|'ll|'d| ?[{L}]+| ?[{N}]+| ?[^{WS}{L}{N}]+|[{WS}]+(?![^{WS}])|[{WS}]+")
print(
    f"Unicode {unicodedata.unidata_version}: the letter class is made of {n_L} contiguous ranges of code points, "
    f"the number class of {n_N}"
)

# the first two are Vietnamese: “I love Vietnam.” and “GPT-2 has 50,257 tokens!”
for s in ["Tôi yêu Việt Nam.", "GPT-2 có 50.257 token!", "it's I'll HOW'S", "  a  b\n\nc"]:
    print(f"{s!r:28} → {PAT.findall(s)}")

Each chunk is a word *with the space in front of it* (`' yêu'`), a run of digits, or a run of punctuation. A few
details worth noting:
- “50.257” is cut into three chunks `' 50'`, `'.'`, `'257'`: the dot that Vietnamese puts between thousands is
  punctuation to GPT-2.
- The suffix `'s` is matched only in lowercase: `HOW'S` becomes `' HOW'`, `"'"`, `'S'`.
- In `'  a  b'`, the first space stands alone and the second goes with the letter: `\s+(?!\S)` leaves the last space
  for the word that follows.

A common approximation replaces `\p{L}` with Python's `[^\W\d_]`. It is close, but Python's `\w` also treats numeric
characters such as ² as letters, `\s` also matches `\x1c`, and the underscore `_` belongs to `\w`, so no class of the
approximate expression matches it. The cell below compares the two ways of cutting:

In [ ]:
NAIVE = re.compile(r"'s|'t|'re|'ve|'m|'ll|'d| ?[^\W\d_]+| ?\d+| ?[^\s\w]+|\s+(?!\S)|\s+")
for s in ["x²+y²", "a\x1c!", "my_var", "Tôi yêu Việt Nam."]:
    a, b = PAT.findall(s), NAIVE.findall(s)
    print(f"{s!r:22} exact:  {a}\n{'':22} approx: {b}  {'(same)' if a == b else '(DIFFERENT)'}")

The two ways of cutting agree on the Vietnamese sentence, but the approximate expression **loses** the `_` of
`my_var` (it matches no class, so it is skipped), and cuts `x²` differently. On text with code, this difference
changes the token sequence completely. From here on we use `PAT`, the exact version; section 6 compares it with
tiktoken, and section 7 compares the total number of tokens on FLORES-200.

## 5 · Learning BPE from scratch

**BPE** (byte-pair encoding; Sennrich et al., 2015, borrowed from a compression algorithm by Gage, 1994) learns a
vocabulary like this:
1. Start with 256 tokens, one per byte. Each chunk is a sequence of byte IDs.
2. Count every **adjacent pair** in every chunk (multiplied by the number of times that chunk occurs).
3. The most frequent pair becomes a new token, with ID 256, 257, …; replace that pair with the new token everywhere.
4. Repeat steps 2–3 until you have as many merges as you want.

The result is an **ordered list of merges**. Here, each byte's ID is its own value (0–255); GPT-2 orders the 256
bytes differently (section 6). When several pairs tie for the most frequent, we take the one seen first.

In [ ]:
def get_stats(words, counts):
    """How often each adjacent pair occurs; each distinct chunk is counted once, times the number of times it occurs."""
    stats = Counter()
    for ids, n in zip(words, counts):
        for pair in itertools.pairwise(ids):
            stats[pair] += n
    return stats


def merge(ids, pair, new_id):
    """Replace every occurrence of the pair `pair` (from left to right) with `new_id`."""
    out, i = [], 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            out.append(new_id)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return out


def train_bpe(chunks, n_merges, log=None):
    """Learn n_merges merges from a list of chunks. Returns the merged pairs (in order) and the table ID → bytes."""
    chunk_counts = Counter(chunks)
    words = [list(c.encode("utf-8")) for c in chunk_counts]
    counts = list(chunk_counts.values())
    merges, vocab = [], {i: bytes([i]) for i in range(256)}
    total = sum(len(w) * n for w, n in zip(words, counts))  # the number of tokens of the whole text
    for k in range(n_merges):
        stats = get_stats(words, counts)
        if not stats:
            break
        pair = max(stats, key=stats.get)  # on a tie, take the pair seen first
        new_id = 256 + k
        merges.append(pair)
        vocab[new_id] = vocab[pair[0]] + vocab[pair[1]]
        merged = [merge(w, pair, new_id) for w in words]
        total -= sum((len(a) - len(b)) * n for a, b, n in zip(words, merged, counts))
        words = merged
        if log is not None:
            log.append((pair, stats[pair], total))
        keep = [i for i, w in enumerate(words) if len(w) > 1]  # a chunk that is already 1 token has no pairs left
        words, counts = [words[i] for i in keep], [counts[i] for i in keep]
    return merges, vocab


def face(b):
    """A token for printing: text if its bytes are valid UTF-8, otherwise the byte codes, e.g. <E1 BB>."""
    try:
        return repr(b.decode("utf-8"))
    except UnicodeDecodeError:
        return "<" + b.hex(" ").upper() + ">"

Let's try it on the first 4 lines of *The Tale of Kiều* (*Truyện Kiều*, by Nguyễn Du). Roughly: “A hundred years in
this world of ours, / talent and destiny are apt to be at odds. / Having lived through an upheaval of seas turned
into mulberry fields, / the things one has seen wrench the heart.”

In [ ]:
KIEU = (
    "Trăm năm trong cõi người ta,\n"
    "Chữ tài chữ mệnh khéo là ghét nhau.\n"
    "Trải qua một cuộc bể dâu,\n"
    "Những điều trông thấy mà đau đớn lòng."
)
kieu_chunks = PAT.findall(KIEU)
n_bytes = len(KIEU.encode("utf-8"))
print(f"{len(KIEU)} characters, {n_bytes} bytes, {len(kieu_chunks)} chunks; the first: {kieu_chunks[:8]}")

### 🤔 Predict first
What is the **first** merge (the most frequent adjacent pair in these 4 lines): a syllable ending like `ng`, an
initial consonant like `th`, or something else?

In [ ]:
log = []
kieu_merges, kieu_vocab = train_bpe(kieu_chunks, 12, log)
print(f"start: {n_bytes} tokens (one token per byte)")
for k, (pair, count, total) in enumerate(log):
    a, b = (kieu_vocab[t] for t in pair)
    print(
        f"merge {k + 1:>2} → token {256 + k}: {face(a):>12} + {face(b):<6} = {face(a + b):<12} ({count} times) → {total} tokens"
    )
assert kieu_vocab[256] == "ữ".encode()[:2] == "ệ".encode()[:2]
assert all(prev - total == count for (_, count, total), prev in zip(log, [n_bytes] + [t for *_, t in log]))

The first merge is the two bytes `E1 BB`: not yet a letter, just the shared beginning of the 3-byte letters ữ, ệ, ộ,
ể, ề, ờ, ớ. BPE doesn't know what a letter or a syllable is; it only counts bytes. Meaningful tokens (`ng`, `' t'`,
`hữ`) appear gradually because they are byte sequences that repeat often. Each merge makes the text shorter by exactly
the number of times the pair occurs (when the occurrences don't overlap, as here): BPE is a compression algorithm.

On 4 lines, only a few pairs occur more than twice, so many pairs tie. We need more text: we learn 1,000 merges on
the 997 Vietnamese sentences of FLORES-200's dev split (this takes a few seconds). The devtest split is kept for the
measurements in section 7.

In [ ]:
vi_merges, vi_vocab = train_bpe([c for s in dev["vi"] for c in PAT.findall(s)], 1000)
print("the first 20 merges:", " ".join(face(vi_vocab[256 + k]) for k in range(20)))
print("the last 10 merges:", " ".join(face(vi_vocab[256 + k]) for k in range(990, 1000)))
partial = [t for t in range(256, 1256) if face(vi_vocab[t]).startswith("<")]
print(f"{1000 - len(partial)} new tokens are whole characters; {len(partial)} are still byte fragments, e.g.", end=" ")
print(*(face(vi_vocab[t]) for t in partial[:4]))

Most new tokens are whole characters (syllables, rhymes, initial consonants…), but there are still byte fragments
such as `E1 BB`: a frequent fragment is merged first, and only then becomes part of longer tokens. We will use this
self-learned merge table in section 7 to compare with GPT-2.

## 6 · GPT-2's real encoder

GPT-2 learned 50,000 merges with the same idea (OpenAI didn't publish its training code), on data that is almost
entirely English. The file `gpt2-bpe.json` stores them as pairs of IDs: merge $i$ (counting from 0) joins two existing
tokens into token $256 + i$. Two small differences from our self-learned version:
- The 256 bytes are numbered in GPT-2's `bytes_to_unicode` order: the printable bytes (`!`…`~`, `¡`…`¬`, `®`…`ÿ`)
  first, then the remaining bytes. That is why the space (byte 32) is token 220 and the newline (byte 10) is token
  198.
- GPT-2's `vocab.json` file writes each byte as a printable character, so the space shows up as `Ġ`: the token
  `' the'` is written `Ġthe`. That is only how the file writes it; there is no `Ġ` character in the text.

In [ ]:
pairs = np.frombuffer(base64.b64decode(bpe_json["merges"]), dtype="<u2").reshape(-1, 2).tolist()
printable = [*range(ord("!"), ord("~") + 1), *range(ord("¡"), ord("¬") + 1), *range(ord("®"), ord("ÿ") + 1)]
BYTE_ORDER = printable + [b for b in range(256) if b not in printable]  # token i < 256 is byte BYTE_ORDER[i]
BYTE_ID = {b: i for i, b in enumerate(BYTE_ORDER)}
RANK = {tuple(p): i for i, p in enumerate(pairs)}  # pair → the merge's rank

TOKEN_BYTES = [bytes([b]) for b in BYTE_ORDER]
for a, b in pairs:  # each merge uses only tokens that come before it
    TOKEN_BYTES.append(TOKEN_BYTES[a] + TOKEN_BYTES[b])

n_vocab = 256 + len(pairs) + 1  # + <|endoftext|>, the special token 50256
print(f"256 bytes + {num(len(pairs), 0)} merges + 1 special token = {num(n_vocab, 0)} tokens")
print("token 220 =", TOKEN_BYTES[220], "· token 198 =", TOKEN_BYTES[198], "· token 50255 =", TOKEN_BYTES[50255])
print("GPT-2's first 10 merges:", " ".join(face(TOKEN_BYTES[256 + i]) for i in range(10)))
print("' the' is token", TOKEN_BYTES.index(b" the"))
assert len(pairs) == 50000 and all(a < 256 + i and b < 256 + i for i, (a, b) in enumerate(pairs))
assert n_vocab == bpe_json["nVocab"] == 50257 and (BYTE_ID[32], BYTE_ID[10]) == (220, 198)

GPT-2's first merge is `' t'`, then `' a'`, `'he'`, `'in'`: traces of English, where “ the” is the most frequent
word (it is token 262, after only 7 merges).

**Encoding** a new chunk: start from the bytes, then repeat: among all adjacent pairs, pick the pair with the **lowest
merge rank** in the table and merge it everywhere; stop when no pair is in the table. This gives exactly the result
of applying the 50,000 merges one after another in the learned order, but takes only a few steps per chunk. It is
also what OpenAI's [encoder.py](https://github.com/openai/gpt-2/blob/master/src/encoder.py) does.

In [ ]:
def encode_chunk(chunk, rank=RANK, byte_id=BYTE_ID, k=math.inf, trace=None):
    """Encode one chunk with the first k merges of the table `rank` (default: all of GPT-2's)."""
    ids = [byte_id[b] for b in chunk.encode("utf-8")]
    while len(ids) > 1:
        pair = min(itertools.pairwise(ids), key=lambda p: rank.get(p, math.inf))
        r = rank.get(pair, math.inf)
        if r >= k:
            break  # no adjacent pair is in (the first k merges of) the table
        ids = merge(ids, pair, 256 + r)
        if trace is not None:
            trace.append((r, ids))
    return ids


CACHE = {}  # each chunk is encoded only once


def encode(text):
    out = []
    for c in PAT.findall(text):
        if c not in CACHE:
            CACHE[c] = encode_chunk(c)
        out += CACHE[c]
    return out


def decode(ids):
    return b"".join(TOKEN_BYTES[t] for t in ids).decode("utf-8", errors="replace")


def show(ids):
    return " | ".join(face(TOKEN_BYTES[t]) for t in ids)


for chunk in [" learning", " Việt"]:
    trace = []
    start = encode_chunk(chunk, k=0)
    encode_chunk(chunk, trace=trace)
    print(f"{chunk!r}: starts as {len(start)} bytes: {show(start)}")
    for r, ids in trace:
        print(f"   merge {num(r, 0):>6} → {show(ids)}   {ids}")

`' learning'` takes 8 merges, at each step the pair with the lowest rank. In `' Việt'`, only the `' Vi'` part is
merged (2 merges), while the three bytes of “ệ” and the letter `t` stand alone: no merge among GPT-2's 50,000 takes
the bytes of “ệ”.

### Checking against tiktoken

OpenAI's [tiktoken](https://github.com/openai/tiktoken) library is the official tokenizer. The IDs below were
computed with `tiktoken.get_encoding("gpt2").encode_ordinary(...)` when this notebook was written, then copied here,
so the notebook doesn't need tiktoken installed. The demo's file also stores 5 pairs of sample sentences with the IDs
tiktoken computed.

In [ ]:
nfd = unicodedata.normalize("NFD", "Tôi yêu Việt Nam.")  # the marks written as separate characters
TIKTOKEN = {  # string → IDs according to tiktoken (the "gpt2" encoding)
    "Hello world": [15496, 995],
    "it's I'll HOW'S": [270, 338, 314, 1183, 29630, 6, 50],
    "  a  b\n\nc": [220, 257, 220, 275, 198, 198, 66],
    "x²+y²": [87, 31185, 10, 88, 31185],
    "a\x1c!": [64, 216, 0],
    "my_var": [1820, 62, 7785],
    "GPT-2 có 50.257 token!": [38, 11571, 12, 17, 269, 10205, 2026, 13, 28676, 11241, 0],
    "Tiếng Việt": [40533, 157, 118, 123, 782, 16049, 157, 119, 229, 83],
    nfd: [2514, 136, 224, 72, 9838, 136, 224, 84, 47154, 136, 96, 136, 224, 83, 17871, 13],  # the NFD form
}
for p in bpe_json["presets"]:
    TIKTOKEN[p["vi"]], TIKTOKEN[p["en"]] = p["viIds"], p["enIds"]

for s, ref in TIKTOKEN.items():
    assert encode(s) == ref, f"differs from tiktoken: {s!r}"
    assert decode(encode(s)) == s  # encoding then decoding must give back exactly the original string
print(f"our encoder == tiktoken on all {len(TIKTOKEN)} strings; examples:")
for s in ["Hello world", "Tôi yêu Việt Nam.", "I love Vietnam."]:
    print(f"  {s!r:22} → {encode(s)}\n  {'':22}   {show(encode(s))}")


def encode_naive(text):  # the same merge table, but chunks cut with the approximate expression
    return [t for c in NAIVE.findall(text) for t in encode_chunk(c)]


wrong = [s for s, ref in TIKTOKEN.items() if encode_naive(s) != ref]
print(f"\nthe approximate expression gives wrong IDs for {len(wrong)} string{'' if len(wrong) == 1 else 's'}: {wrong}")

Our encoder gives exactly tiktoken's IDs on every test string, both the demo's sample sentences and the tricky cases
(double spaces, `'s`, control characters, the NFD form). The approximate chunking is wrong on `my_var`; on `x²+y²`
and `a\x1c!` it cuts differently but happens to give the same IDs, because no merge joins `x` with `²`. Section 7
runs the encoder on the English and Vietnamese sentences of FLORES-200's devtest split and compares the total number
of tokens with the count from tiktoken.

## 7 · How many tokens does Vietnamese cost?

### 🤔 Predict first
The Vietnamese version of FLORES-200 has 38.99% more bytes than the English one. With GPT-2's tokenizer, how many
times as many tokens does it take: about 1.4 times, like the bytes, or far more?

In [ ]:
n_tok = {lang: sum(len(encode(s)) for s in test[lang]) for lang in ("en", "vi")}
units = {lang: sum(len(s.split()) for s in test[lang]) for lang in ("en", "vi")}  # words / syllables: split on spaces
for lang, name, unit in (("en", "English", "word"), ("vi", "Vietnamese", "syllable")):
    print(
        f"{name:10}: {num(n_tok[lang], 0):>7} tokens for {num(units[lang], 0)} {unit}s"
        f" → {num(n_tok[lang] / units[lang], 2)} tokens per {unit}"
    )
ratio = n_tok["vi"] / n_tok["en"]
print(
    f"Vietnamese takes {num(ratio, 2)} times as many tokens, even though it has only "
    f"{pct(n_byte['vi'] / n_byte['en'] - 1)} more bytes"
)
print(
    f"GPT-2's 1,024-token context window holds about {num(1024 * units['en'] / n_tok['en'], 0)} English words, "
    f"but only {num(1024 * units['vi'] / n_tok['vi'], 0)} Vietnamese syllables"
)
gpt2_count = next(e for e in bpe_json["flores"]["encodings"] if e["id"] == "gpt2")  # counted by tiktoken, in the file
assert (n_tok["en"], n_tok["vi"]) == (gpt2_count["en"], gpt2_count["vi"]) == (27044, 122278), "differs from tiktoken"
assert units == bpe_json["flores"]["units"]

**4.52 times**, more than three times the gap in bytes, and the total number of tokens matches exactly the count
tiktoken gives on the 1,012 sentence pairs. An English word takes 1.23 tokens on average, a Vietnamese syllable 3.98
tokens (written Vietnamese puts a space between syllables, so splitting on spaces counts syllables): almost every
common English word has a token of its own, while Vietnamese syllables are chopped into pieces. In the same 1,024-token
context window, GPT-2 can “see” 829 English words but only 257 Vietnamese syllables; a service that charges per token
with this tokenizer would also charge that many times as much for Vietnamese (Petrov et al., 2023, call this the
unfairness of tokenizers between languages).

Why? Merges go only to pairs that are frequent in the text the tokenizer was learned on, and that text is almost
entirely English: the GPT-2 paper says the authors *deliberately removed non-English pages* from WebText, and the
first 10 merges in section 6 are all pieces of English words. So the bytes of Vietnamese are hardly ever merged.
The two measurements below show it: the share of tokens that are only **byte fragments** (not a whole character), and
the number of tokens of each word or syllable.

In [ ]:
def is_partial(t):
    """A token that is only part of a character: its bytes are not valid UTF-8."""
    try:
        TOKEN_BYTES[t].decode("utf-8")
        return False
    except UnicodeDecodeError:
        return True


chunk_counts = {lang: Counter(c for s in test[lang] for c in PAT.findall(s)) for lang in ("en", "vi")}
for lang in ("en", "vi"):
    share = sum(is_partial(t) for s in test[lang] for t in encode(s)) / n_tok[lang]
    print(f"{lang}: {pct(share)} of the tokens are byte fragments")
    assert abs(share - bpe_json["flores"]["partial"][lang]) < 6e-5

word = re.compile(rf" ?[{L}]+")  # chunks made only of letters: one word / syllable, with the space in front
hist = {}
for lang in ("en", "vi"):
    h = [0] * 6  # 1, 2, …, 5, ≥ 6 tokens
    for c, n in chunk_counts[lang].items():
        if word.fullmatch(c):
            h[min(len(CACHE[c]), 6) - 1] += n
    hist[lang] = np.array(h)
    print(
        f"{lang}: {num(sum(h), 0)} words/syllables; how many take 1…5, ≥ 6 tokens: {h} → "
        f"{pct(h[0] / sum(h))} need just 1 token"
    )
assert {k: v.tolist() for k, v in hist.items()} == bpe_json["flores"]["chunkHist"]

fig, ax = plt.subplots(figsize=(8, 3.4))
x = np.arange(6)
for k, (lang, label, color, hatch) in enumerate(
    [("en", "English (per word)", BLUE, None), ("vi", "Vietnamese (per syllable)", ORANGE, "//")]
):
    share = hist[lang] / hist[lang].sum()
    ax.bar(x + (k - 0.5) * 0.38, 100 * share, 0.38, color=color, hatch=hatch, edgecolor="white", label=label)
ax.set_xticks(x, ["1", "2", "3", "4", "5", "≥ 6"])
ax.set(xlabel="tokens GPT-2 needs for one word / syllable", ylabel="share (%)")
ax.set_title("FLORES-200 devtest: the number of tokens of each word / syllable", fontsize=11)
ax.legend(frameon=False)
fmt_axes(fig)
plt.show()

Almost half of the Vietnamese tokens (48.04%) are only byte fragments, versus 0.27% in English. In the figure, the
blue bars (English) pile up at 1 token (94.28% of words); the hatched orange bars (Vietnamese) spread from 2 to more
than 6 tokens, and only 6.28% of syllables are a single token.

### The cost of diacritics
How many tokens does each Vietnamese letter take on its own? The figure below shows the 12 vowels × 6 tones (each
Vietnamese vowel can carry one of 5 tone marks, or none), plus the letter đ: the number in each cell is the number
of tokens, and the darker the cell, the higher the cost.

In [ ]:
VOWELS = ["a", "ă", "â", "e", "ê", "i", "o", "ô", "ơ", "u", "ư", "y"]
TONES = {  # the 6 tones (in Vietnamese: ngang, huyền, sắc, hỏi, ngã, nặng), named after their marks
    "no mark": "",
    "grave": chr(0x300),
    "acute": chr(0x301),
    "hook": chr(0x309),
    "tilde": chr(0x303),
    "dot below": chr(0x323),
}  # tone marks in combining form
letters = [[unicodedata.normalize("NFC", v + m) for m in TONES.values()] for v in VOWELS]
cost = np.array([[len(encode_chunk(ch)) for ch in row] for row in letters])
n_utf8 = np.array([[len(ch.encode("utf-8")) for ch in row] for row in letters])
cost_d = len(encode_chunk("đ"))

fig, ax = plt.subplots(figsize=(6.2, 6.4))
grid = np.vstack([cost, [cost_d] + [0] * 5])
ax.imshow(grid, cmap=LinearSegmentedColormap.from_list("orange", ["#ffffff", ORANGE]), vmin=0, vmax=3.6)
for r, row in enumerate([*letters, ["đ"]]):
    for c, ch in enumerate(row):
        ax.text(c, r, f"{ch} {grid[r, c]}", ha="center", va="center", fontsize=11)
ax.set_xticks(range(6), list(TONES), fontsize=9, rotation=30)
ax.set_yticks(range(13), [*VOWELS, "đ"])
ax.tick_params(length=0)
ax.set_title("Tokens GPT-2 needs for each letter on its own", fontsize=11)
for side in ("left", "bottom"):
    ax.spines[side].set_visible(False)
plt.show()

all_cost = [*cost.ravel(), cost_d]
print("letters that take 1 / 2 / 3 tokens:", [all_cost.count(k) for k in (1, 2, 3)])
print("1 token:", " ".join(ch for row, k in zip(letters, cost) for ch, c in zip(row, k) if c == 1))
print(
    f"{(n_utf8 == 3).sum()} letters are 3 bytes long in UTF-8; their token counts: {sorted(set(cost[n_utf8 == 3].tolist()))}"
)
assert [all_cost.count(k) for k in (1, 2, 3)] == [17, 11, 45] and (cost[n_utf8 == 3] == 3).all()

17 letters take only 1 token: the 6 vowels without diacritics (1 byte) and 11 two-byte Latin-1 letters such as á, à,
ê, ô, which are common in French, Spanish and Portuguese. 11 letters take 2 (two UTF-8 bytes that are not merged),
and **all 45 letters that are 3 bytes long take exactly 3 tokens**: one token per byte.

### Why exactly 3 tokens?
The cell below counts, among all 50,257 GPT-2 tokens, the tokens that contain the leading byte pair `E1 BA` or
`E1 BB` of the 3-byte letters.

In [ ]:
lead = [t for t, b in enumerate(TOKEN_BYTES) if b"\xe1\xba" in b or b"\xe1\xbb" in b]
print("tokens that contain E1 BA or E1 BB:", len(lead))
assert lead == []

tails = {}  # 3-byte letters (capitals included) whose last two bytes are one token
for cp in range(0x1EA0, 0x1EFA):
    tail = chr(cp).encode("utf-8")[1:]
    if tail in TOKEN_BYTES:
        tails[chr(cp)] = TOKEN_BYTES.index(tail)
print("3-byte letters whose last two bytes are one token:", {ch: (t, face(TOKEN_BYTES[t])) for ch, t in tails.items()})
print(f"'Ồ' → {show(encode_chunk('Ồ'))}  ·  'ồ' → {show(encode_chunk('ồ'))}")

spread = Counter()  # in real sentences: how many tokens each 3-byte letter of devtest lies on
for c, n in chunk_counts["vi"].items():
    ends = list(itertools.accumulate(len(TOKEN_BYTES[t]) for t in CACHE[c]))
    spans = list(zip([0, *ends[:-1]], ends))  # the byte positions of each token in the chunk
    p = 0
    for ch in c:
        k = len(ch.encode("utf-8"))
        if 0x1EA0 <= ord(ch) <= 0x1EF9:
            spread[sum(a < p + k and b > p for a, b in spans)] += n
        p += k
print(
    f"FLORES-200 devtest: {num(spread.total(), 0)} three-byte letters; "
    f"the number of tokens each letter lies on: {dict(spread)}"
)
assert set(spread) == {3}

**Not a single token**: the shared two leading bytes of the 90 letters ạ…ỹ (capitals included) have never been
merged with each other, so each of these letters takes at least 2 tokens. There is just one exception that takes 2
instead of 3: the capital **Ồ** (`E1 BB 92`), because the pair `BB 92` happens to be a token. That pair is the end of
the Chinese/Japanese character 黒 (`E9 BB 92`), which has nothing to do with Vietnamese. In the real sentences of
FLORES-200, every 3-byte letter lies on 3 tokens (the byte `E1` sometimes shares a token with the space in front of
it, `' \xe1'` is token 28053, but the letter still spans 3 tokens).

### A tokenizer learned on Vietnamese
This cost comes not from Vietnamese itself but from the *data* the tokenizer was learned on. We compare three merge
tables on the same devtest split: the table self-learned on Vietnamese in section 5, a table of 300 merges
self-learned on the English dev split (a few seconds), and GPT-2's table, each using only its first $k$ merges.
Because BPE learns greedily, one step at a time, the first $k$ merges of a run that learns 1,000 merges are exactly
the result of a run that learns only $k$ merges.

In [ ]:
en_merges, _ = train_bpe([c for s in dev["en"] for c in PAT.findall(s)], 300)
IDENTITY = {b: b for b in range(256)}  # the self-learned tables: a byte's ID is its own value
TOKENIZERS = {
    "self-learned, Vietnamese": ({p: i for i, p in enumerate(vi_merges)}, IDENTITY, [0, 30, 100, 300, 1000]),
    "self-learned, English": ({p: i for i, p in enumerate(en_merges)}, IDENTITY, [0, 30, 100, 300]),
    "GPT-2": (RANK, BYTE_ID, [0, 30, 100, 300, 1000, 3000, 10000, 50000]),
}


def count_tokens(lang, rank, byte_id, k):
    return sum(len(encode_chunk(c, rank, byte_id, k)) * n for c, n in chunk_counts[lang].items())


curves = {}
for name, (rank, byte_id, ks) in TOKENIZERS.items():
    curves[name] = [(k, count_tokens("en", rank, byte_id, k), count_tokens("vi", rank, byte_id, k)) for k in ks]
    for k, en, vi in curves[name]:
        print(
            f"{name:24} k = {num(k, 0):>6}: English {num(en, 0):>7} · Vietnamese {num(vi, 0):>7} → ×{num(vi / en, 2)}"
        )
assert curves["GPT-2"][-1][1:] == (27044, 122278) and curves["GPT-2"][0][1:] == (n_byte["en"], n_byte["vi"])

The figure below plots the same numbers: the horizontal axis is the number of merges $k$ (log scale), the vertical
axis is the ratio of Vietnamese to English tokens. The dotted line at 1 means “both languages cost the same”; at
$k = 0$ (bytes only) the ratio is 1.39, exactly the byte ratio.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
styles = {
    "self-learned, Vietnamese": (GREEN, "o", "-"),
    "self-learned, English": (BLUE, "s", "--"),
    "GPT-2": (ORANGE, "^", "-"),
}
for name, pts in curves.items():
    color, marker, ls = styles[name]
    k = [max(p[0], 1) for p in pts]  # k = 0 is plotted at 1 on the log scale
    ax.plot(k, [p[2] / p[1] for p in pts], color=color, marker=marker, ls=ls, lw=2, label=name)
ax.axhline(1, color=GREY, lw=1, ls=":")
ax.set_xscale("log")
ax.set(xlabel="number of merges k (first point: k = 0)", ylabel="Vietnamese / English tokens")
ax.set_title("Vietnamese/English token ratio on FLORES-200 devtest, by merge table")
ax.legend(frameon=False)
fmt_axes(fig)
plt.show()

The table self-learned on English follows GPT-2's curve almost exactly: learned on English, the more merges there
are, the worse off Vietnamese is. The table self-learned on 997 Vietnamese sentences does the opposite: with just
1,000 merges, the Vietnamese version already takes fewer tokens than the English one (×0.54). OpenAI's newer
tokenizers have much larger vocabularies, and the gap narrows (counts by tiktoken, stored in the demo's file):

In [ ]:
for e in bpe_json["flores"]["encodings"]:
    print(
        f"{e['label']:20} vocabulary {num(e['vocab'], 0):>7} tokens: Vietnamese / English = ×{num(e['vi'] / e['en'], 2)}"
    )

## 8 · Token → vector: the embedding table

A token ID is just a name: token 5822 is not “bigger” than token 582. A neural network needs a vector for each
token. GPT-2 keeps an **embedding table** $E$ with $V = 50{,}257$ rows and $d = 768$ columns: token number $i$ takes
row $i$.

$$x = \operatorname{onehot}(i)\,E, \qquad E \in \mathbb{R}^{V \times d}$$

Multiplying the one-hot vector (all 0s, except a 1 at position $i$) by $E$ is exactly picking out row $i$, so in code
this is the lookup `E[i]`, not a multiplication. It is also a linear layer $z = xW$ as in Lesson 03, except that the
input is always one-hot. The numbers in $E$ are learned by backpropagation like any other weights (Lesson 06). At the
input layer, the gradient flows back only to the rows of the tokens that appear in the batch; but GPT-2 reuses the
same $E$ in the output layer (section 11), so through the softmax every row gets a gradient at every step.

We have 137 real rows of $E$ (from the demo's file): each row is stored as int8 (integers −127…127) with a scale
factor. Decompressing means multiplying the two.

In [ ]:
words = [w["w"] for w in emb["words"]]
ids = [w["id"] for w in emb["words"]]
q = np.frombuffer(base64.b64decode(emb["rows"]), dtype=np.int8).reshape(len(words), emb["dim"])
E_rows = q.astype(np.float64) * np.array(emb["scales"])[:, None]  # 137 × 768: GPT-2's rows E[id]
W = {w: i for i, w in enumerate(words)}  # word → position in E_rows

V, d = emb["nVocab"], emb["dim"]
print(f"E has {num(V, 0)} × {d} = {num(V * d, 0)} parameters ({num(V * d * 4 / 1e6, 0)} MB in float32)")
print(f"we have {len(words)} rows, each with {d} numbers")
print(f"the small GPT-2 has 124,439,808 parameters → E is {pct(V * d / 124_439_808, 1)} of them")
assert V * d == emb["params"] == 38_597_376 and q.shape == (137, 768)

# each word's id is exactly its token (with the space in front) according to the encoder of section 6
assert all(encode(" " + w) == [i] for w, i in zip(words, ids))
i = W["king"]
print(
    f"' king' → token {encode(' king')} → row {ids[i]} of E; the first 8 numbers:",
    " ".join(num(v, 3) for v in E_rows[i, :8]),
)
print(f"length ‖E[{ids[i]}]‖ = {num(np.linalg.norm(E_rows[i]), 2)}")

The figure below draws three rows of $E$, each with its 768 numbers arranged in a 12 × 64 grid: blue is positive,
orange is negative, darker is farther from 0. Nobody wrote these numbers by hand: they were learned while GPT-2
practiced predicting the next token.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9, 4.6))
vmax = np.abs(E_rows[[W["king"], W["queen"], W["Monday"]]]).max()
for ax, w in zip(axes, ["king", "queen", "Monday"]):
    im = ax.imshow(E_rows[W[w]].reshape(12, 64), cmap=DIVERGING, vmin=-vmax, vmax=vmax)
    ax.set_title(f"' {w}': token {ids[W[w]]}, row {ids[W[w]]} of E", fontsize=10)
    ax.axis("off")
fig.colorbar(im, ax=axes, fraction=0.03)
fmt_axes(fig)
plt.show()

### Cosine similarity
To compare two vectors, we measure the angle between them: $\cos(u, v) = \dfrac{u \cdot v}{\lVert u \rVert\, \lVert v
\rVert}$, which is 1 when they point the same way, 0 when they are perpendicular, −1 when they point in opposite
directions. Divide each row by its length first (unit vectors), and the cosine is just a dot product, and the whole
cosine matrix is one matrix multiplication.

### 🤔 Predict first
“king” and “Monday” have nothing to do with each other. Is the cosine between their rows close to 0, clearly
positive, or negative?

In [ ]:
U = E_rows / np.linalg.norm(E_rows, axis=1, keepdims=True)  # every row has length 1
S = U @ U.T  # 137 × 137: S[i, j] = cos(row i, row j)
for a, b in [("king", "queen"), ("man", "woman"), ("France", "Paris"), ("king", "apple"), ("king", "Monday")]:
    print(f"cos({a!r}, {b!r}) = {num(S[W[a], W[b]], 3)}")
off = ~np.eye(len(words), dtype=bool)
print(f"\naverage over all pairs among the 137 words: {num(S[off].mean(), 3)}")
r = emb["random"]  # measured on 199,994 random token pairs from the whole vocabulary (the demo's file)
print(
    f"average over {num(r['pairs'], 0)} random token pairs from the whole vocabulary: "
    f"{num(r['mean'], 4)} ± {num(r['sd'], 4)}"
)

Clearly positive: 0.241. All the rows of $E$ lean toward one common direction, so even two randomly chosen tokens have
an average cosine of 0.2681. This phenomenon is called **anisotropy** (Ethayarajh, 2019). That is why a cosine means
something only in *comparison*: king/queen's 0.657 is notable only because it is well above the 0.27 baseline.

Two ways to check this baseline: the first shows that it comes from training, the second that it is a shared
direction:
- **At initialization**, GPT-2 draws each number of $E$ from $\mathcal N(0,\ 0.02^2)$. Two random 768-dimensional
  vectors are almost perpendicular: their cosine is around 0, with a standard deviation of about $1/\sqrt{768}$.
- **Subtract the mean vector** of the rows, and the common direction disappears: unrelated pairs go back to around
  0.

In [ ]:
rng = np.random.default_rng(0)
E_init = rng.normal(0, 0.02, size=E_rows.shape)  # simulate 137 rows at initialization
U_init = E_init / np.linalg.norm(E_init, axis=1, keepdims=True)
S_init = U_init @ U_init.T
print(
    f"at initialization: mean cos {num(S_init[off].mean(), 3)} ± {num(S_init[off].std(), 3)}"
    f" (1/√768 = {num(1 / math.sqrt(768), 3)})"
)

C_rows = E_rows - E_rows.mean(0)  # subtract the mean vector of the 137 rows
Uc = C_rows / np.linalg.norm(C_rows, axis=1, keepdims=True)
Sc = Uc @ Uc.T
for a, b in [("king", "queen"), ("king", "Monday")]:
    print(
        f"after subtracting the mean: cos({a!r}, {b!r}) = {num(Sc[W[a], W[b]], 3)}  (before: {num(S[W[a], W[b]], 3)})"
    )
print(f"after subtracting the mean: mean cos over all pairs {num(Sc[off].mean(), 3)}")
assert abs(S_init[off].mean()) < 0.01 and S[off].mean() > 0.2 and S[W["king"], W["Monday"]] > 0.2

A cosine map of 16 words from 4 groups shows both points. On the left is the raw cosine; on the right, the cosine
after subtracting the mean. Blue is positive, orange is negative, on the same −1…1 scale; the labels on both axes are
the words, with lines between the groups.

In [ ]:
group = ["king", "queen", "man", "woman", "France", "Paris", "Japan", "Tokyo"]
group += ["Monday", "Tuesday", "seven", "eight", "go", "went", "walk", "walked"]
gi = [W[w] for w in group]
fig, axes = plt.subplots(1, 2, figsize=(10, 4.9))
for ax, M, title in [(axes[0], S, "raw cosine"), (axes[1], Sc, "after subtracting the mean vector")]:
    im = ax.imshow(M[np.ix_(gi, gi)], cmap=DIVERGING, vmin=-1, vmax=1)
    ax.set_xticks(range(16), group, rotation=90, fontsize=8)
    ax.set_yticks(range(16), group, fontsize=8)
    for k in (3.5, 7.5, 11.5):
        ax.axhline(k, color="black", lw=0.8)
        ax.axvline(k, color="black", lw=0.8)
    ax.set_title(title, fontsize=10)
fig.colorbar(im, ax=axes, fraction=0.025)
fmt_axes(fig)
plt.show()

On the left, almost every cell is blue: anisotropy's 0.27 baseline. On the right, the four blocks on the diagonal are
still blue (family and gender, countries and capitals, days and numbers, verbs and their past tense), while the pairs
from different groups turn white or orange.

### Nearest neighbors
The demo's file stores the 10 nearest neighbors of each word among **all 50,257 tokens** (computed in float64 on the
full table). We print the lists of a few words, and check the neighbors that are among our 137 rows: the cosine
recomputed from the int8 rows must match the stored cosine to within 0.003.

In [ ]:
strings = emb["strings"]  # ID → text of the token, for the tokens the file mentions
pos = {t: k for k, t in enumerate(ids)}  # ID → position among the 137 rows
err = 0.0
for k, nb in enumerate(emb["neighbours"]):
    for t, c in nb:
        if t in pos:
            err = max(err, abs(S[k, pos[t]] - c))
print(f"largest difference between the recomputed cosine (int8) and the stored cosine (float64): {num(err, 4)} < 0.003")
assert err < 0.003

for w in ["king", "Monday", "seven", "Paris"]:
    nb = emb["neighbours"][W[w]][:6]
    print(f"{w!r:9} → " + ", ".join(f"{strings[str(t)]!r} {num(c, 2)}" for t, c in nb))

The nearest neighbors show that embeddings group together tokens that appear in similar contexts: days of the week,
numbers, cities. The lists also contain capitalized variants and variants without the space in front (`'Paris'`,
`'King'`): to GPT-2 these are different tokens, each with its own row, and the model has to learn by itself that
they mean nearly the same thing.

## 9 · Analogies: $b - a + c$

Mikolov et al. (2013) found that the difference of two word vectors can carry a relation: vector(“King”) −
vector(“Man”) + vector(“Woman”) is closest to vector(“Queen”). The method: take the unit vectors of the three words,
compute $v = \hat b - \hat a + \hat c$ (here $a$ = man, $b$ = woman, $c$ = king), then find the word with the largest
cosine with $v$. A detail that is often overlooked: the word2vec paper says that they **exclude the question words**
from the search.

### 🤔 Predict first
With the real rows of $E$ in GPT-2, if we do *not* exclude the three input words, which word is closest to $v$?

In [ ]:
def analogy(a, b, c, exclude=True, top=5):
    """“a is to b as c is to ?”: rank the 137 words by their cosine with b̂ − â + ĉ."""
    v = U[W[b]] - U[W[a]] + U[W[c]]
    s = U @ (v / np.linalg.norm(v))
    order = [k for k in np.argsort(-s) if not (exclude and words[k] in (a, b, c))]
    return [(words[k], float(s[k])) for k in order[:top]]


keep, drop = analogy("man", "woman", "king", exclude=False), analogy("man", "woman", "king", exclude=True)
print("keeping the input words:", ", ".join(f"{w} {num(c, 3)}" for w, c in keep))
print("excluding the input words:", ", ".join(f"{w} {num(c, 3)}" for w, c in drop))

relations = {r["id"]: r for r in emb["relations"]}
stored = next(r for r in relations["gender"]["results"] if (r["a"], r["b"], r["c"]) == ("man", "woman", "king"))
for key, name in (("incl", "keeping"), ("excl", "excluding")):
    print(f"all 50,257 tokens, {name}:", ", ".join(f"{strings[str(t)]!r} {num(c, 3)}" for t, c in stored[key]))
    in_ours = [t for t, _ in stored[key] if t in pos]  # the tokens of the stored list that are among our 137 rows
    ours = analogy("man", "woman", "king", exclude=key == "excl", top=len(in_ours))
    assert [ids[W[w]] for w, _ in ours] == in_ours  # same order
assert keep[0][0] == "king" and keep[1][0] == "queen" and drop[0][0] == "queen"

The closest is **“king” itself** (0.740); “queen” comes second (0.694), and moves up to rank 1 only when we exclude
the three input words. The list over the whole vocabulary gives the same result. The reason is length:
$v = \hat c + (\hat b - \hat a)$ contains the whole unit vector of “king”, while the offset $\hat b - \hat a$ is
shorter, because “man” and “woman” are already close to each other:

In [ ]:
offset = U[W["woman"]] - U[W["man"]]
print(f"‖woman − man‖ = {num(np.linalg.norm(offset), 3)}  versus ‖king‖ = 1 (a unit vector)")
print(
    f"cos(man, woman) = {num(S[W['man'], W['woman']], 3)} → ‖woman − man‖ = √(2 − 2·cos) = {num(math.sqrt(2 - 2 * S[W['man'], W['woman']]), 3)}"
)

So “king − man + woman = queen” is half true: the “male → female” direction is real (it pushes “queen” above
“prince” and “kings”), but the arithmetic wins only when we forbid the most obvious answer. Nissim et al. (2020)
point out the flip side of this rule: for “man : doctor = woman : ?”, the search is *not allowed* to answer
“doctor”, even when that is the right answer, so the forced results (“nurse”…) have been read as evidence of bias.

The demo's file runs every question built from the pairs of 6 relations (580 questions) on the whole vocabulary. We
recompute the same 580 questions on the 137 int8 rows and check: every cosine score in the stored lists, if that
token is among the 137 rows, must match the score we recompute.

In [ ]:
err, n_q, top_is_input, top_is_c = 0.0, 0, 0, 0
for r in emb["relations"]:
    for res in r["results"]:
        a, b, c = res["a"], res["b"], res["c"]
        v = U[W[b]] - U[W[a]] + U[W[c]]
        s = U @ (v / np.linalg.norm(v))
        for t, score in res["incl"] + res["excl"]:
            if t in pos:
                err = max(err, abs(s[pos[t]] - score))
        top_is_input += words[int(np.argmax(s))] in (a, b, c)
        top_is_c += words[int(np.argmax(s))] == c
        n_q += 1
print(f"{n_q} questions; largest score difference from the file: {num(err, 4)}")
idof = dict(zip(words, ids))
full = sum(r["incl"][0][0] in {idof[r[k]] for k in "abc"} for rel in emb["relations"] for r in rel["results"])
print("without excluding the input words, rank 1 is one of the three input words:")
print(
    f"  among 137 words: {top_is_input}/{n_q} questions ({pct(top_is_input / n_q)}), of which {top_is_c} are c itself"
)
print(f"  among all 50,257 tokens (file): {full}/{n_q} questions ({pct(full / n_q)})")
assert n_q == sum(r["n"] for r in emb["relations"]) == 580 and err < 0.003

Without excluding the input words, about 7 answers in 10 are one of the question's three words, at almost the same
rate among the 137 words and over the whole vocabulary. It is almost always $c$ itself (“king” in the example above),
because $v$ contains the whole $\hat c$.

### Results by relation, and a baseline
The table below counts the share of questions whose correct answer is at rank 1, under three ways of scoring:
- **keep** and **exclude** the three input words;
- **only c**: ignore $a$ and $b$, and answer with the word closest to $c$ (still excluding the three input words).
  Linzen (2016) proposed this baseline: if it is already right often, most of the analogy's “success” comes from the
  answer already lying close to $c$.

Each way of scoring is computed twice: on our 137 words (few candidates, computed right here), and on all 50,257
tokens (read from the demo's file).

In [ ]:
def score(rel, pool_scores):
    """The share right at rank 1 for one relation: keep / exclude / only c, on the 137 words."""
    hits = np.zeros(3)
    for res in rel["results"]:
        a, b, c, d = (W[res[k]] for k in "abcd")
        v = U[b] - U[a] + U[c]
        s = U @ (v / np.linalg.norm(v))
        hits[0] += np.argmax(s) == d
        s[[a, b, c]] = -np.inf
        hits[1] += np.argmax(s) == d
        sc = pool_scores[c].copy()
        sc[[a, b, c]] = -np.inf
        hits[2] += np.argmax(sc) == d
    return hits / rel["n"]


REL_EN = {  # English names of the 6 relations (the file's labels are the Vietnamese demo's)
    "gender": "Male → female",
    "capital": "Country → capital",
    "nationality": "Country → nationality",
    "plural": "Singular → plural",
    "comparative": "Adjective → comparative",
    "past": "Present → past",
}
print(
    f"{'relation':24}{'n':>5} │ {'137 words: keep · excl · only c':>31} │ {'50,257 tokens: keep · excl · only c':>35}"
)
for r in emb["relations"]:
    ours = score(r, S)
    theirs = (r["incl"], r["excl"], r["onlyC"])
    print(
        f"{REL_EN[r['id']]:24}{r['n']:>5} │ {' · '.join(pct(x, 0) for x in ours):>31} │ {' · '.join(pct(x, 0) for x in theirs):>35}"
    )
g = emb["google"]
print(
    f"\nGoogle question set (Mikolov et al., 2013), {num(g['n'], 0)}/{num(g['total'], 0)} questions whose 4 words are "
    f"all tokens, whole vocabulary: keep {pct(g['incl'])} · exclude {pct(g['excl'])} · only c {pct(g['onlyC'])}"
)

Three things to read from the table:
- **Excluding the input words** changes the results the most: on the Google question set, from 11.15% to 78.76%.
- **The “only c” baseline** is already right 32.26% of the time on the Google set, and for plurals and the past tense
  it is 100% right on the 137 words: “cats” is already the word closest to “cat”. The analogy only really adds
  information when it far exceeds this baseline (as for capitals).
- **The candidate set** matters: with 137 candidates, accuracy is much higher than over the whole vocabulary
  (capitals: 86% versus 62%), because there are fewer distracting words. Two numbers can be compared only when they
  are scored on the same set.

### The “country → capital” direction
If a relation is a fixed direction, the arrows from each country to its capital should be nearly parallel. The
figure below projects the 22 words of this relation down to 2 dimensions with PCA (their two directions of largest
variance; Lesson 19 compares PCA with autoencoders): blue circles are countries, orange squares are capitals, and
gray arrows join each pair.

In [ ]:
pairs_cap = relations["capital"]["pairs"]
cap_words = [w for pair in pairs_cap for w in pair]
X = U[[W[w] for w in cap_words]]
X = X - X.mean(0)
_, sv, Vt = np.linalg.svd(X, full_matrices=False)
P2 = X @ Vt[:2].T  # coordinates on the two principal directions
P2 *= np.sign((P2[1::2] - P2[::2]).mean(0))  # the sign varies by machine: flip so the mean arrow points up and right
xy = dict(zip(cap_words, P2))
LABEL = {  # (dx, dy, alignment) in points, for labels that sit close together; other words use the default position
    "Spain": (6, 10, "left"), "France": (10, -1, "left"), "Italy": (6, -8, "left"), "Germany": (6, -4, "left"),
    "Rome": (-6, -2, "right"), "Madrid": (0, 8, "center"), "Paris": (5, -2, "left"),
    "Tokyo": (-6, 0, "right"), "Beijing": (0, 8, "center"), "Seoul": (0, -14, "center"), "Bangkok": (6, 0, "left"),
}  # fmt: skip
fig, ax = plt.subplots(figsize=(7.5, 5.2))
for country, city in pairs_cap:
    ax.annotate("", xy[city], xy[country], arrowprops={"arrowstyle": "->", "color": GREY, "lw": 1.2})
    ax.scatter(*xy[country], marker="o", s=45, color=BLUE, zorder=3)
    ax.scatter(*xy[city], marker="s", s=45, color=ORANGE, zorder=3)
    for w, default in ((country, (4, 4, "left")), (city, (4, -10, "left"))):
        dx, dy, ha = LABEL.get(w, default)
        ax.annotate(w, xy[w], xytext=(dx, dy), textcoords="offset points", fontsize=8, ha=ha, va="center")
ax.set_xlim(right=P2[:, 0].max() + 0.12)  # leave room for the labels on the right
ax.set(xlabel="principal direction 1", ylabel="principal direction 2")
ax.set_title("Country (circle) → capital (square): 11 pairs, PCA projection", fontsize=11)
fmt_axes(fig)
plt.show()
print(
    f"the two principal directions keep {pct((sv[:2] ** 2).sum() / (sv**2).sum(), 1)} of the variance of the 22 vectors"
)

rng = np.random.default_rng(1)
for rid in ("capital", "gender", "plural", "past"):
    D = np.array([U[W[b]] - U[W[a]] for a, b in relations[rid]["pairs"]])  # each row: the difference of one pair
    D /= np.linalg.norm(D, axis=1, keepdims=True)
    C = D @ D.T
    print(f"{REL_EN[rid]:24}: mean cos between the differences b̂ − â = {num(C[np.triu_indices(len(D), 1)].mean(), 2)}")
R = rng.integers(0, len(words), size=(200, 2))
R = R[R[:, 0] != R[:, 1]]
D = U[R[:, 1]] - U[R[:, 0]]
D /= np.linalg.norm(D, axis=1, keepdims=True)
print(f"{'differences of random word pairs':24}: {num((D @ D.T)[np.triu_indices(len(D), 1)].mean(), 2)}")

The table printed the accuracy of each relation, both on the 137 words and on the whole vocabulary. The projection
shows that the country → capital arrows all point the same way, but are not perfectly parallel. The cosines say the
same: the differences $\hat b - \hat a$ of two pairs in the same relation have an average cosine of 0.31–0.48, while
the differences of two random word pairs are around 0. The relation's direction is real, but it is only part of each
difference.

## 10 · Try changing things

The cells below rerun the same functions with different inputs. You can edit the values and run them again.

### Your own sentence

In [ ]:
my_text = "Học sâu bắt đầu từ những con số."  # edit this sentence (Vietnamese for “Deep learning starts from numbers.”)
my_ids = encode(my_text)
print(f"{len(my_text)} characters, {len(my_text.encode('utf-8'))} bytes → {len(my_ids)} tokens")
print(my_ids)
print(show(my_ids))
assert decode(my_ids) == my_text

### Unicode forms and diacritics
The same 1,012 Vietnamese sentences, written in decomposed form (NFD: the marks are separate characters) or with all
diacritics removed (đ → d):

In [ ]:
def strip_marks(s):
    """Remove the diacritics: split the marks off (NFD), delete the combining marks, turn đ into d."""
    s = unicodedata.normalize("NFD", s.replace("đ", "d").replace("Đ", "D"))
    return unicodedata.normalize("NFC", "".join(ch for ch in s if not unicodedata.combining(ch)))


print(strip_marks("Tôi yêu Việt Nam. Đà Nẵng"))
forms = {
    "precomposed (NFC)": lambda s: unicodedata.normalize("NFC", s),
    "decomposed (NFD)": lambda s: unicodedata.normalize("NFD", s),
    "no diacritics": strip_marks,
}
for name, f in forms.items():
    n = sum(len(encode(f(s))) for s in test["vi"])
    print(f"{name:17}: {num(n, 0):>7} tokens → ×{num(n / n_tok['en'], 2)} versus English")

FLORES-200 is written in NFC, so the first line matches section 7. In NFD, each mark becomes a separate 2-byte
character and the number of tokens goes up further. With all diacritics removed, Vietnamese costs only about twice as
much as English: more than half the cost lies in the bytes of the diacritics. But removing the diacritics destroys
meaning (“ma, má, mà, mả, mã, mạ”, which mean ghost, mother, but, grave, horse, rice seedling, all become the same
word), so the right fix is a merge table learned on Vietnamese, not editing the text.

### A bilingual tokenizer
Learn 300 merges on both the 997 English and the 997 Vietnamese sentences of the dev split (a few seconds). Try
changing `n_mix`:

In [ ]:
n_mix = 300
mix_merges, _ = train_bpe([c for lang in ("en", "vi") for s in dev[lang] for c in PAT.findall(s)], n_mix)
mix_rank = {p: i for i, p in enumerate(mix_merges)}
en_n, vi_n = (count_tokens(lang, mix_rank, IDENTITY, n_mix) for lang in ("en", "vi"))
print(f"bilingual, {n_mix} merges: English {num(en_n, 0)} · Vietnamese {num(vi_n, 0)} → ×{num(vi_n / en_n, 2)}")
for name in ("self-learned, Vietnamese", "self-learned, English"):
    k, en, vi = next(p for p in curves[name] if p[0] == 300)
    print(f"versus {name}, 300 merges: ×{num(vi / en, 2)}")

### Other analogies
Each line prints the top 3 words when keeping and when excluding the input words. Every word must be among the
file's 137 words (`print(words)` to see the list).

In [ ]:
for a, b, c in [
    ("France", "Paris", "Japan"),
    ("big", "bigger", "small"),
    ("go", "went", "see"),
    ("man", "king", "woman"),
]:
    keep = ", ".join(w for w, _ in analogy(a, b, c, exclude=False, top=3))
    drop = ", ".join(w for w, _ in analogy(a, b, c, top=3))
    print(f"{b} − {a} + {c}: keep → {keep:28} exclude → {drop}")

## 11 · Limits and next steps

At this point, a sentence has become a sequence of vectors: tokenize, then look up one row of $E$ for each token.
The table $E$ has two big limits.

**Embeddings are static.** A token's row is the same in every sentence, in every context. Four Vietnamese syllables,
“hay” (good; or), “cho” (give; for), “con” (child), “ra” (out), happen to be GPT-2 tokens because they are also
English words (or pieces of words), so their rows carry English meanings: “hay” is dried grass, and “cho” sits next to
“helicopter” (chopper).

In [ ]:
for w in ["hay", "cho"]:
    nb = emb["neighbours"][W[w]][:6]
    print(f"{w!r} (a Vietnamese syllable, token {ids[W[w]]}) → " + ", ".join(f"{strings[str(t)]!r}" for t, _ in nb))

**Embeddings don't know order.** Two sentences with opposite meanings can use exactly the same set of tokens. If the
model only adds up (or averages) their vectors, the two sentences give the same vector:

In [ ]:
# the Vietnamese pair means “dog bites man” and “man bites dog”
for pair in [(" chó cắn người", " người cắn chó"), (" cat ate mouse", " mouse ate cat")]:
    t1, t2 = (encode(s) for s in pair)
    print(f"{pair[0]!r}: {t1}\n{pair[1]!r}: {t2}\n  same set of tokens: {sorted(t1) == sorted(t2)}")
    assert sorted(t1) == sorted(t2) and t1 != t2
rows = {s: E_rows[[pos[t] for t in encode(s)]] for s in (" cat ate mouse", " mouse ate cat")}
m1, m2 = (r.mean(0) for r in rows.values())
print(f"mean embedding of the two English sentences: largest difference {np.abs(m1 - m2).max():.0e} (rounding only)")
assert np.allclose(m1, m2)

To know who bit whom, the model has to read the vectors **in order**, and a token has to be understood through the
tokens around it. Lesson 15 does that with a recurrent network, which reads one token at a time and carries a state
along. Lessons 16–17 do it with the attention mechanism; GPT-2 also adds a position vector $P[t]$ to each token (the
table $P$ has 1,024 × 768 numbers, Lesson 17). At the output, GPT-2 reuses the same table $E$: the score of the next
token is the dot product of the final state with each row of $E$.

## Summary
- Text is UTF-8 bytes: 256 bytes can read any string, so GPT-2 never needs an “unknown” token. Vietnamese letters
  with diacritics take 2–3 bytes; the Vietnamese version of FLORES-200 has 5.02% more characters than the English
  one but 38.99% more bytes.
- The pre-tokenizer cuts a sentence into chunks (words with the space in front, numbers, punctuation); merges don't
  cross chunk boundaries. The exact version of `\p{L}`/`\p{N}` gives exactly tiktoken's IDs; the `\w` approximation
  loses the `_`.
- BPE learns by repeating “merge the most frequent adjacent pair”, and encodes by reapplying the merges in the
  learned order. GPT-2's vocabulary = 256 bytes + 50,000 merges + 1 special token = 50,257.
- On FLORES-200, Vietnamese takes 4.52 times as many tokens (3.98 tokens per syllable; 48.04% of the tokens are byte
  fragments), because the merge table was learned from English text. No token contains `E1 BA` or `E1 BB`, so the 45
  three-byte letters always take 3 tokens. A table of 1,000 merges learned on Vietnamese reverses the ratio (×0.54).
- An embedding is a table lookup: token $i$ takes row $i$ of $E$ (50,257 × 768 = 38,597,376 parameters, 31.0% of
  the small GPT-2).
- A cosine means something only against the baseline: two arbitrary tokens have an average cos of 0.2681
  (anisotropy), while at initialization it is 0.
- king − man + woman: the closest is “king” itself; “queen” comes first only when the three input words are excluded
  (Google set: 11.15% → 78.76%).

## Review questions
1. Why does GPT-2's byte-level tokenizer not need an “unknown” token (`[UNK]`), even though it was learned almost
   only on English?
2. The first merge learned on the 4 lines of Kiều is `E1 BB`, not a letter. Why can BPE create a token like that?
3. When encoding, why must the merges be applied in exactly the learned order, rather than choosing whichever split
   gives the fewest tokens?
4. The Vietnamese version has only 38.99% more bytes but takes 4.52 times as many tokens. Where does the difference
   come from? How does it affect GPT-2's 1,024-token context window?
5. cos(“king”, “Monday”) = 0.241. Should we conclude that these two words are “somewhat similar in meaning”? What
   should it be compared with?
6. Why is the vector closest to $\widehat{\text{woman}} - \widehat{\text{man}} + \widehat{\text{king}}$ “king”
   itself?

<details><summary>Hints and answers</summary>

1. The first 256 tokens are all the bytes, and every Unicode string is a sequence of UTF-8 bytes. An unfamiliar
   character just takes more tokens (one token per byte); it is never lost.
2. BPE only counts adjacent byte pairs; it doesn't know where letters begin or end. `E1 BB` is the shared beginning
   of many 3-byte letters (ữ, ệ, ộ…), so it is the most frequent pair even though it is not yet a letter.
3. The model was trained on exactly that way of splitting: each token has a row of $E$ learned from the contexts it
   appears in. Splitting differently (even with fewer tokens) would feed in tokens that the model has hardly ever
   seen in that position. Applying the merges in order also guarantees that a string always gives the same sequence
   of tokens.
4. From the merge table: the merges were learned on English text, so almost half of the Vietnamese tokens are only
   byte fragments (48.04% versus 0.27%). The 1,024-token window holds about 829 English words but only 257
   Vietnamese syllables.
5. No: two random tokens already have an average cos of 0.2681, because every row of $E$ leans toward one common
   direction. 0.241 is right at the baseline; after subtracting the mean vector, it becomes −0.033. You have to
   compare with the baseline (or subtract the mean) first.
6. That vector contains the whole unit vector of “king”, while the offset woman − man is shorter (length 0.845,
   because man and woman are already close to each other), so it is still closest to “king”. The word2vec paper
   excludes the question words from the search.
</details>

## Further reading
- R. Sennrich, B. Haddow, A. Birch (2015), [Neural Machine Translation of Rare Words with Subword Units](https://arxiv.org/abs/1508.07909):
  BPE for neural networks.
- A. Radford et al. (2019), [Language Models are Unsupervised Multitask Learners](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf):
  section 2.2 on byte-level BPE and the pre-tokenizer; the original code, [encoder.py](https://github.com/openai/gpt-2/blob/master/src/encoder.py).
- A. Karpathy, [Let's build the GPT Tokenizer](https://www.youtube.com/watch?v=zduSFxRajkE) (video) and
  [minbpe](https://github.com/karpathy/minbpe): rebuilding the same algorithm, with exercises.
- Hugging Face LLM Course, [Byte-Pair Encoding tokenization](https://huggingface.co/learn/llm-course/chapter6/5).
- OpenAI, [tiktoken](https://github.com/openai/tiktoken): the official tokenizer that the notebook matches.
- A. Petrov et al. (2023), [Language Model Tokenizers Introduce Unfairness Between Languages](https://arxiv.org/abs/2305.15425).
- NLLB Team (2022), [No Language Left Behind](https://arxiv.org/abs/2207.04672): the FLORES-200 parallel sentences.
- T. Mikolov et al. (2013), [Efficient Estimation of Word Representations in Vector Space](https://arxiv.org/abs/1301.3781),
  and T. Mikolov, W. Yih, G. Zweig (2013), [Linguistic Regularities in Continuous Space Word Representations](https://aclanthology.org/N13-1090/).
- T. Linzen (2016), [Issues in evaluating semantic spaces using word analogies](https://arxiv.org/abs/1606.07736):
  the “only c” baseline.
- M. Nissim, R. van Noord, R. van der Goot (2020), [Fair is Better than Sensational: Man is to Doctor as Woman is to Doctor](https://arxiv.org/abs/1905.09866).
- K. Ethayarajh (2019), [How Contextual are Contextualized Word Representations?](https://arxiv.org/abs/1909.00512):
  anisotropy in GPT-2's embeddings.

**Next lesson:** [Lesson 15 · RNN & LSTM](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/rnn/): reading a sequence
of vectors in order, carrying a state along.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")